# 🎨 Notebook 1: LinkedIn (professional network) - Class Design

Design the core of a **professional network**: profiles, connections, jobs, applications.
We walk through common modeling **mistakes** with runnable examples and fix each one step by step.
Notebook 2 then assembles the "best" choices into a working implementation.

## 🛠️ Setup

```bash
cd 07-object-oriented-design/linkedin
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` -> **Reload Window**.


## What we are designing

LinkedIn is a *professional* network. The core differs from Facebook in a few important ways:

- **Connections are explicit**: you can't just "add" someone - you *send a request*, and the other side *accepts* or *rejects*.
- **Connections are symmetric** once accepted (unlike Twitter follows).
- Users have **Profiles** with work history, education, and skills.
- Companies post **Jobs**; users **apply** with a status lifecycle.

### Domain model (ASCII)

```
   User --o-- Profile --*-- Experience, Education, Skill
    |
    | sends connection_request
    v
   ConnectionRequest (PENDING -> ACCEPTED | REJECTED)
    |  on ACCEPTED: mirror edge into both users' connections
    v
   User  <-- connections (symmetric set) -->  User

   Company --posts--> Job --*-- required_skills
                      ^
                      | applies
   User -------------> Application (SUBMITTED -> REVIEWED -> OFFER | REJECTED)
```

### 6 design decisions we will justify with code

1. **ConnectionRequest is a first-class object** - not just a `friends: set`. We need to model the pending state.
2. **Statuses are `Enum`s** - not free-form strings - so typos and invalid values fail fast.
3. **State transitions are guarded methods** (`accept`, `reject`) - not raw assignments to `.status`.
4. **Skill matching uses set intersection**, not list scans - clearer and O(min(m, n)).
5. **Stable integer IDs** for hashing - so `User`, `Job`, etc., can live in `set`s and `dict` keys safely.
6. **Endorsements are a separate edge** between users - not a counter on a skill - so we know *who* endorsed *whom* for *what*.

## 🚦 Bad -> Best progression

Below are **five common mistakes** when modeling LinkedIn. Each has a runnable snippet showing the problem, then a runnable fix.

### ❌ Bad #1 - Modeling connections as a plain `set[User]` (like Facebook)

*Naive*: `alice.connections.add(bob)` and done. But LinkedIn has a **request/accept flow** - you cannot just self-add
someone to their connections list. We lose the ability to represent a *pending* request or to reject one.

In [ ]:
class BadUser:
    def __init__(self, name):
        self.name = name
        self.connections = set()

    def connect(self, other):
        # BUG: self-connecting without consent from 'other'
        self.connections.add(other)
        other.connections.add(self)

a = BadUser("Alice"); b = BadUser("Bob")
a.connect(b)
print("Alice just force-connected to Bob, no acceptance step:",
      b in a.connections and a in b.connections)

### ✅ Fix - A `ConnectionRequest` object with an explicit status

Model the **request itself** as an entity. Only after the receiver calls `accept()` do both users' connection sets get updated.

In [ ]:
from enum import Enum

class ReqStatus(Enum):
    PENDING  = "pending"
    ACCEPTED = "accepted"
    REJECTED = "rejected"

class GoodUser:
    def __init__(self, name):
        self.name = name
        self.connections = set()
    def __repr__(self): return f"User({self.name})"
    def __hash__(self): return id(self)

class ConnectionRequest:
    def __init__(self, sender, receiver):
        self.sender, self.receiver = sender, receiver
        self.status = ReqStatus.PENDING

    def accept(self):
        self.status = ReqStatus.ACCEPTED
        self.sender.connections.add(self.receiver)
        self.receiver.connections.add(self.sender)   # symmetric

    def reject(self):
        self.status = ReqStatus.REJECTED

alice, bob = GoodUser("Alice"), GoodUser("Bob")
req = ConnectionRequest(alice, bob)
print("before accept:", alice.connections, bob.connections)
req.accept()
print("after accept :", alice.connections, bob.connections)
print("status       :", req.status)

### ❌ Bad #2 - Storing status as a free-form string

*Naive*: `req.status = "acepted"` (typo). Python happily accepts the string - your `if status == "accepted"` check silently fails.

In [ ]:
class BadReq:
    def __init__(self): self.status = "pending"

r = BadReq()
r.status = "acepted"   # typo, no error
if r.status == "accepted":
    print("accepted!")
else:
    print("bug: typo in status was not detected, got:", r.status)

### ✅ Fix - Enums fail fast on typos

An `Enum` only allows the members you declared. Typos become `AttributeError` at the place the bug happens.

In [ ]:
try:
    ReqStatus.ACEPTED   # typo
except AttributeError as e:
    print("caught typo at the source:", e)

### ❌ Bad #3 - Raw status assignment allows impossible transitions

*Naive*: expose `req.status` for direct writes. Now code anywhere can rewind `ACCEPTED -> PENDING`, accept a rejected request,
or accept the same request twice - each "accept" re-adding the connection (idempotent here, but imagine a side effect like
"send acceptance email" firing twice).

In [ ]:
class LooseReq:
    def __init__(self, s, r):
        self.sender, self.receiver = s, r
        self.status = ReqStatus.PENDING

emails_sent = []
def send_email(to, msg): emails_sent.append((to.name, msg))

r = LooseReq(alice, bob)
r.status = ReqStatus.ACCEPTED
send_email(r.sender, "your request was accepted")
r.status = ReqStatus.PENDING        # illegal rewind, no one stopped us
r.status = ReqStatus.ACCEPTED       # accept "again"
send_email(r.sender, "your request was accepted")
print("emails sent:", emails_sent, "  <- duplicate!")

### ✅ Fix - Guarded transitions: `accept()` / `reject()` check the current state

Encapsulate state changes in methods that validate the current state first. This is a tiny **state machine**:

```
PENDING --accept()--> ACCEPTED    (terminal)
PENDING --reject()--> REJECTED    (terminal)
```

Any other transition raises an error, so bugs are loud and local.

In [ ]:
class SafeRequest:
    def __init__(self, sender, receiver):
        self.sender, self.receiver = sender, receiver
        self.status = ReqStatus.PENDING

    def accept(self):
        if self.status is not ReqStatus.PENDING:
            raise ValueError(f"cannot accept from {self.status}")
        self.status = ReqStatus.ACCEPTED
        self.sender.connections.add(self.receiver)
        self.receiver.connections.add(self.sender)

    def reject(self):
        if self.status is not ReqStatus.PENDING:
            raise ValueError(f"cannot reject from {self.status}")
        self.status = ReqStatus.REJECTED

carol = GoodUser("Carol")
r = SafeRequest(alice, carol)
r.accept()
try:
    r.accept()                       # guarded, raises
except ValueError as e:
    print("second accept blocked:", e)

### ❌ Bad #4 - Skill matching with nested list loops

*Naive*: given a job's `required_skills` as a list and a user's `skills` as a list, loop through one and check membership in the other.
It works, but it's O(m*n), and the intent ("how many required skills does this user have?") is hidden in the loop.

In [ ]:
required = ["python", "sql", "kafka"]
alice_skills = ["python", "sql", "redis"]

matched = 0
for rs in required:
    for us in alice_skills:          # nested loop = O(m*n)
        if rs == us:
            matched += 1
            break
print("nested-loop matches:", matched, "/", len(required))

### ✅ Fix - Use `set` intersection

Sets give O(1) membership and `&` expresses the question directly: *which required skills does the user have?*

In [ ]:
required_set = {"python", "sql", "kafka"}
alice_set   = {"python", "sql", "redis"}
matched = required_set & alice_set
score = len(matched) / len(required_set)
print("matches:", matched, "  score:", score)

### ❌ Bad #5 - Endorsements as a counter on a `Skill`

*Naive*: `skill.endorsements += 1`. Now we cannot answer "did Bob endorse Alice for Python?" or prevent Bob from endorsing
the same skill 10 times. We have lost the graph edge.

In [ ]:
class BadSkill:
    def __init__(self, name):
        self.name = name
        self.endorsements = 0

py = BadSkill("python")
py.endorsements += 1   # bob endorses
py.endorsements += 1   # bob endorses AGAIN, counted again (bug)
py.endorsements += 1   # carol endorses
print("endorsements:", py.endorsements, "  but we don't know WHO endorsed")

### ✅ Fix - Endorsements as `(endorser, endorsee, skill)` edges

Store them in a set keyed by `(endorser_id, endorsee_id, skill)`. That makes a double-endorsement a no-op, and we can
answer "who endorsed Alice for Python?" in one filter.

In [ ]:
endorsements: set[tuple[int,int,str]] = set()

def endorse(endorser, endorsee, skill):
    endorsements.add((id(endorser), id(endorsee), skill))

endorse(bob, alice, "python")
endorse(bob, alice, "python")        # duplicate, set dedupes
endorse(carol, alice, "python")

def count_endorsements(user, skill):
    return sum(1 for (_, e, s) in endorsements if e == id(user) and s == skill)

print("Alice endorsements for python:", count_endorsements(alice, "python"), " <- 2, not 3")

## 📋 Final class table

| Class | Role | Key fields |
|---|---|---|
| `User` | person in the network | `id`, `name`, `profile`, `connections: set[User]` |
| `Profile` | a user's public info | `headline`, `experiences`, `education`, `skills` |
| `Experience` | one past/current job | `title`, `company`, `start_year`, `end_year` |
| `ConnectionRequest` | pending/accepted/rejected request | `sender`, `receiver`, `status` |
| `ReqStatus` | enum | `PENDING, ACCEPTED, REJECTED` |
| `Company` | posts jobs | `name`, `employees` |
| `Job` | a posting | `title`, `company`, `description`, `required_skills` |
| `Application` | user applies to job | `user`, `job`, `status` |
| `AppStatus` | enum | `SUBMITTED, REVIEWED, OFFER, REJECTED` |
| `Endorsement` | `(endorser, endorsee, skill)` edge | - |
| `Message` / `Inbox` | DMs between connections | `sender`, `receiver`, `text`, `ts` |

➡️ Continue to **Notebook 2** for the runnable implementation with the connection state machine, applications, endorsements, messaging, and a simple job recommender.